# NB07 — Hasil Akhir HDMC dan Temuan Kunci

Notebook laporan final, **tidak melatih model apapun**, cuma membaca file hasil
yang sudah tersimpan (dari nb02b, nb03, nb05, nb06) dan menyusunnya jadi satu
gambaran utuh. Aman dijalankan ulang kapan saja, cepat, tidak makan waktu training.

Isi:
1. Konsolidasi HDMC + 10 pembanding, 4 dataset -> satu file Excel
2. Peringkat HDMC dan ringkasan uji signifikansi per dataset
3. Pola keunggulan HDMC terhadap Imbalance Ratio (klaim utama naskah)
4. Kontribusi meta-learner (ringkasan hasil ablasi nb05)
5. Dampak penggantian backbone CNN-BiLSTM -> CNN-BiGRU (Part 2 vs Part 3)

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATASET_ORDER, TAB_DIR
print("Urutan dataset:", DATASET_ORDER)

In [ ]:
### Redirect output ke results_clean/ (tidak menimpa results/ asli)
import os
RESULTS_CLEAN_ROOT = os.path.abspath(os.path.join("..", "results_clean"))
for _sub in ["tables", "figures", "models", "predictions"]:
    os.makedirs(os.path.join(RESULTS_CLEAN_ROOT, _sub), exist_ok=True)

_new_tab_dir = os.path.join(RESULTS_CLEAN_ROOT, "tables")
_new_mdl_dir = os.path.join(RESULTS_CLEAN_ROOT, "models")
_new_pred_dir = os.path.join(RESULTS_CLEAN_ROOT, "predictions")
_new_fig_dir = os.path.join(RESULTS_CLEAN_ROOT, "figures")

import src.hdmc_multiview as _m1
import src.comparator_runner as _m2
import src.experiment_runner as _m3
import src.justification_runner as _m4
import src.data_quality as _m5

for _mod in [_m1, _m2, _m3, _m4, _m5]:
    if hasattr(_mod, "TAB_DIR"):
        _mod.TAB_DIR = _new_tab_dir
    if hasattr(_mod, "MDL_DIR"):
        _mod.MDL_DIR = _new_mdl_dir
    if hasattr(_mod, "PRED_DIR"):
        _mod.PRED_DIR = _new_pred_dir

TAB_DIR = _new_tab_dir
MDL_DIR = _new_mdl_dir
PRED_DIR = _new_pred_dir
FIG_DIR = _new_fig_dir

print(f"Output diarahkan ke: {RESULTS_CLEAN_ROOT}")

## 1. Konsolidasi seluruh hasil -> satu file Excel

Menggabungkan `<dataset>_hdmc_final_metrics.json` (HDMC) dan
`comparator_<dataset>.json` (10-11 pembanding: 5 eksternal + 5-6 baseline)
untuk keempat dataset, jadi satu `DataFrame` lalu satu file Excel dengan
beberapa sheet (ringkasan semua, per dataset, uji signifikansi).

**Termasuk waktu training dan inference** (`train_time_s`, `infer_ms_per_sample`
pada batch size 1, skenario deployment real-time satu sampel). Untuk pembanding
biasa ini diambil langsung dari `agg["timing"]`. **Untuk HDMC lebih rumit**:
pipeline-nya melatih 3 backbone (satu per treatment) + 1 meta-classifier,
jadi total waktu training/inference HDMC dihitung sebagai **jumlah** waktu
ketiga backbone tersebut ditambah meta-classifier (bukan cuma salah satu
komponennya), supaya perbandingan terhadap pembanding lain adil (pembanding
cuma 1 model, HDMC pipeline lengkap 4 komponen).

**Nilai terbaik per dataset per metrik di-bold** di file Excel-nya (tertinggi
untuk macro-F1/BAcc/MCC/akurasi, tercepat/terendah untuk waktu training dan
inference).

Tidak ada angka yang dihitung ulang dari model (semua diambil dari file yang
sudah tersimpan), cuma dijumlah/disusun ulang.

In [ ]:
CATEGORY_MAP = {
    "HDMC": "Proposed",
    "Abdelli-BiGRU": "Eksternal", "Soothar-CNNLSTM": "Eksternal",
    "Yang-IMCBiLSTM": "Eksternal", "Healy-GRUClassifier": "Eksternal",
    "Titouni-CNNExtractor": "Eksternal",
    "Baseline-CNNOnly": "Baseline", "Baseline-BiGRUOnly": "Baseline",
    "Baseline-BiLSTMOnly": "Baseline", "Baseline-LogReg": "Baseline",
    "Baseline-RandomForest": "Baseline", "Baseline-SVM": "Baseline",
}

def ambil_timing_pembanding(agg):
    """Pembanding biasa: satu model, timing langsung dari agg['timing'].
    Pakai 'or {}' (bukan cuma default di .get), karena kalau key "timing"
    ADA tapi isinya None (bukan hilang), .get(k, {}) tetap balikin None.

    Dua pola nama key inference berbeda di kode (bukan bug data, cuma
    penamaan beda antar fungsi): model deep learning (Abdelli/Soothar/Yang/
    Healy/CNNOnly/BiGRUOnly/BiLSTMOnly) lewat inference_timing_sweep() ->
    "t_infer_bs1_per_sample_ms_mean" (banyak batch size, bs1 dipakai di sini).
    Model klasik (LogReg/RandomForest/SVM) dan Titouni (run_titouni_ensemble)
    -> "t_infer_per_sample_ms_mean" (satu angka, tanpa embel-embel batch
    size). Coba yang pertama dulu, fallback ke yang kedua."""
    t = agg.get("timing") or {}
    train_s = t.get("t_train_s_mean", np.nan)
    infer_ms = t.get("t_infer_bs1_per_sample_ms_mean", t.get("t_infer_per_sample_ms_mean", np.nan))
    return train_s, infer_ms

def ambil_timing_hdmc(agg):
    """HDMC: jumlahkan waktu 3 backbone (oof_timing_per_treatment) + meta-classifier."""
    train_s_total, infer_ms_total = 0.0, 0.0
    ada_data = False

    oof_timing = agg.get("oof_timing_per_treatment") or {}
    for label, info in oof_timing.items():
        t_agg = (info or {}).get("agg") or {}
        if "t_train_s_mean" in t_agg:
            train_s_total += t_agg["t_train_s_mean"]
            ada_data = True
        if "t_infer_bs1_per_sample_ms_mean" in t_agg:
            infer_ms_total += t_agg["t_infer_bs1_per_sample_ms_mean"]

    meta_t = agg.get("timing") or {}
    if "t_train_s_mean" in meta_t:
        train_s_total += meta_t["t_train_s_mean"]
        ada_data = True
    if "t_infer_per_sample_ms_mean" in meta_t:
        infer_ms_total += meta_t["t_infer_per_sample_ms_mean"]

    if not ada_data:
        return np.nan, np.nan
    return train_s_total, infer_ms_total

def ambil_metrik(agg, dataset, model, kategori):
    """Ambil metrik evaluasi + timing dari dict agg, toleran kalau ada key
    yang tidak tersedia (pakai NaN, tidak crash)."""
    g = lambda k: agg.get(k, np.nan)
    if model == "HDMC":
        train_s, infer_ms = ambil_timing_hdmc(agg)
    else:
        train_s, infer_ms = ambil_timing_pembanding(agg)
    return {
        "dataset": dataset, "model": model, "kategori": kategori,
        "macro_f1_mean": g("macro_f1_mean"), "macro_f1_std": g("macro_f1_std"),
        "bacc_mean": g("bacc_mean"), "bacc_std": g("bacc_std"),
        "mcc_mean": g("mcc_mean"), "mcc_std": g("mcc_std"),
        "accuracy_mean": g("accuracy_mean"), "accuracy_std": g("accuracy_std"),
        "train_time_s": train_s, "infer_ms_per_sample": infer_ms,
        "n_observations": g("n_observations"),
    }

rows_all = []
for ds in DATASET_ORDER:
    hdmc_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
    with open(hdmc_path) as f:
        hdmc = json.load(f)
    rows_all.append(ambil_metrik(hdmc, ds, "HDMC", "Proposed"))

    comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
    with open(comp_path) as f:
        comparators = json.load(f)
    for model_name, agg in comparators.items():
        if model_name == "Baseline-BiLSTMOnly":
            continue  # tidak dipakai (HDMC sekarang CNN-BiGRU, BiLSTMOnly cuma sisa lama khusus SOR)
        kategori = CATEGORY_MAP.get(model_name, "Lainnya")
        rows_all.append(ambil_metrik(agg, ds, model_name, kategori))

df_semua = pd.DataFrame(rows_all)
df_semua = df_semua.sort_values(["dataset", "macro_f1_mean"], ascending=[True, False]).reset_index(drop=True)

sig_path = os.path.join(TAB_DIR, "uji_signifikansi_hdmc_vs_semua_pembanding.csv")
df_sig = pd.read_csv(sig_path) if os.path.exists(sig_path) else pd.DataFrame()

print(f"Total baris: {len(df_semua)} ({df_semua['dataset'].nunique()} dataset)")
df_semua

## 1b. Metrik per kelas — digabung ke sheet masing-masing dataset

F1 tiap kelas (bukan cuma macro-F1) dijadikan **kolom tambahan** di sheet
per-dataset yang sudah ada (bukan sheet terpisah), supaya semua angka satu
model kelihatan sejajar dalam satu baris. Nama kolom pakai **nama kelas
asli** dataset itu:

- OTDR: 8 kolom (Normal, Bending Event, Dirty Connector, Bad Splice, Fiber
  Cut, PC Connector, Reflector, Fiber Tapping)
- SOR: 6 kolom (Normal, Air Gap, Bad Splice, Bending, Dirty Connector,
  Fiber Cut)
- HardFailure & SoftFailure: 2 kolom saja (Normal, Failure), sesuai
  jumlah kelas aslinya (biner)

Coba beberapa kemungkinan nama key dari file JSON (struktur bisa beda
antar fungsi), kalau tidak ketemu, dicetak diagnostik jelas supaya bisa
diperbaiki cepat.

In [ ]:
KEMUNGKINAN_KEY_PER_KELAS = ["per_class", "per_class_f1", "f1_per_class",
                              "per_class_f1_mean", "f1_per_class_mean"]
KEMUNGKINAN_SUBKEY_F1 = ["f1_mean", "f1", "f1_score", "F1", "f1-score"]
KELAS_BUKAN_KELAS_ASLI = {"accuracy", "macro avg", "weighted avg", "micro avg"}

def cari_key_per_kelas(agg):
    for k in KEMUNGKINAN_KEY_PER_KELAS:
        if k in agg and isinstance(agg[k], dict) and len(agg[k]) > 0:
            return k
    return None

def ekstrak_nilai_f1(nilai):
    """nilai bisa berupa angka langsung, atau dict bersarang
    {precision, recall, f1, support}. Ekstrak F1-nya saja."""
    if isinstance(nilai, (int, float)):
        return float(nilai)
    if isinstance(nilai, dict):
        for sub_k in KEMUNGKINAN_SUBKEY_F1:
            if sub_k in nilai:
                return float(nilai[sub_k])
    return None

per_kelas_records = []  # (dataset, model, kelas, f1)
key_ditemukan_contoh = None

def tambah_per_kelas(ds, model_name, agg):
    global key_ditemukan_contoh
    k = cari_key_per_kelas(agg)
    if not k:
        return
    key_ditemukan_contoh = key_ditemukan_contoh or k
    for kelas, nilai in agg[k].items():
        if kelas in KELAS_BUKAN_KELAS_ASLI:
            continue  # bukan nama kelas asli, lewati (mis. "accuracy", "macro avg")
        f1 = ekstrak_nilai_f1(nilai)
        if f1 is not None:
            per_kelas_records.append((ds, model_name, kelas, f1))

for ds in DATASET_ORDER:
    hdmc_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
    with open(hdmc_path) as f:
        hdmc = json.load(f)
    tambah_per_kelas(ds, "HDMC", hdmc)

    comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
    with open(comp_path) as f:
        comparators = json.load(f)
    for model_name, agg in comparators.items():
        if model_name == "Baseline-BiLSTMOnly":
            continue
        tambah_per_kelas(ds, model_name, agg)

if per_kelas_records:
    df_pk_long = pd.DataFrame(per_kelas_records, columns=["dataset", "model", "kelas", "f1"])
    print(f"Key ditemukan: '{key_ditemukan_contoh}'")
    print(f"Total baris per-kelas terbaca: {len(df_pk_long)}")

    # Gabungkan ke df_semua: per dataset, pivot kelas jadi kolom, lalu merge
    daftar_per_dataset = []
    for ds in DATASET_ORDER:
        sub_semua = df_semua[df_semua["dataset"] == ds]
        sub_pk = df_pk_long[df_pk_long["dataset"] == ds]
        if not sub_pk.empty:
            pivot = sub_pk.pivot_table(index="model", columns="kelas", values="f1", aggfunc="first")
            pivot.columns = [f"F1_{c}" for c in pivot.columns]
            gabung = sub_semua.merge(pivot, on="model", how="left")
        else:
            gabung = sub_semua.copy()
        daftar_per_dataset.append(gabung)
    df_semua_lengkap = pd.concat(daftar_per_dataset, ignore_index=True)
    kolom_kelas = [c for c in df_semua_lengkap.columns if c.startswith("F1_")]
    print(f"Kolom kelas ditambahkan: {kolom_kelas}")
else:
    df_semua_lengkap = df_semua.copy()
    kolom_kelas = []
    print("[TIDAK KETEMU] Tidak ada key per-kelas yang cocok dari daftar:")
    print(f"  {KEMUNGKINAN_KEY_PER_KELAS}")
    print("Cek manual struktur file JSON (mis. otdr_hdmc_final_metrics.json)")
    print("untuk lihat nama key sebenarnya, kasih tahu saya untuk diperbaiki.")

df_semua_lengkap.head(3)

In [ ]:
from openpyxl.styles import Font

KOLOM_TINGGI_LEBIH_BAIK = ["macro_f1_mean", "bacc_mean", "mcc_mean", "accuracy_mean"] + kolom_kelas
KOLOM_RENDAH_LEBIH_BAIK = ["train_time_s", "infer_ms_per_sample"]

out_path = os.path.join(TAB_DIR, "HASIL_KONSOLIDASI_HDMC_vs_PEMBANDING.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df_semua_lengkap.to_excel(writer, sheet_name="Ringkasan Semua", index=False)
    for ds in DATASET_ORDER:
        # dropna(axis=1, how="all"): buang kolom kelas dataset LAIN yang ikut
        # kebawa (mis. F1_Failure di sheet Otdr), karena semua barisnya NaN.
        sheet_df = df_semua_lengkap[df_semua_lengkap["dataset"] == ds].dropna(axis=1, how="all")
        sheet_df.to_excel(writer, sheet_name=ds.capitalize(), index=False)
    if not df_sig.empty:
        df_sig.to_excel(writer, sheet_name="Uji Signifikansi", index=False)

from openpyxl import load_workbook
wb = load_workbook(out_path)
for ds in DATASET_ORDER:
    sheet_name = ds.capitalize()
    if sheet_name not in wb.sheetnames:
        continue
    ws = wb[sheet_name]
    sub = df_semua_lengkap[df_semua_lengkap["dataset"] == ds].dropna(axis=1, how="all").reset_index(drop=True)
    header = [cell.value for cell in ws[1]]

    for kolom in KOLOM_TINGGI_LEBIH_BAIK:
        if kolom not in header or kolom not in sub.columns or sub[kolom].isna().all():
            continue
        col_idx = header.index(kolom) + 1
        best_row_idx = sub[kolom].idxmax() + 2
        ws.cell(row=best_row_idx, column=col_idx).font = Font(bold=True)

    for kolom in KOLOM_RENDAH_LEBIH_BAIK:
        if kolom not in header or kolom not in sub.columns or sub[kolom].isna().all():
            continue
        col_idx = header.index(kolom) + 1
        best_row_idx = sub[kolom].idxmin() + 2
        ws.cell(row=best_row_idx, column=col_idx).font = Font(bold=True)

wb.save(out_path)
print(f"Tersimpan (kolom kelas dirapikan per dataset, nilai terbaik di-bold): {out_path}")


## 2. Peringkat HDMC dan ringkasan uji signifikansi per dataset

Untuk tiap dataset: urutan macro-F1 semua model (HDMC ada di posisi berapa),
dan dari uji Wilcoxon+Cliff's delta (`nb03`/`nb06`), berapa pembanding yang
HDMC menangkan signifikan vs berapa yang setara (negligible effect, bukan
kekalahan, cuma tidak terbukti beda secara statistik).

In [ ]:
print("="*78)
print("PERINGKAT HDMC DI TIAP DATASET (berdasar macro-F1)")
print("="*78)
peringkat_rows = []
for ds in DATASET_ORDER:
    sub = df_semua[df_semua["dataset"] == ds].reset_index(drop=True)
    peringkat_hdmc = sub[sub["model"] == "HDMC"].index[0] + 1
    total = len(sub)
    selisih_ke_2 = sub.loc[0, "macro_f1_mean"] - sub.loc[1, "macro_f1_mean"] if peringkat_hdmc == 1 else None
    print(f"\n[{ds.upper()}] HDMC peringkat {peringkat_hdmc}/{total}")
    print(sub[["model", "kategori", "macro_f1_mean", "bacc_mean", "mcc_mean"]].to_string(index=False))
    if selisih_ke_2 is not None:
        print(f"  -> Selisih ke posisi 2: +{selisih_ke_2:.4f}")
    peringkat_rows.append({
        "dataset": ds, "peringkat_HDMC": peringkat_hdmc, "total_model": total,
        "selisih_ke_posisi_2": round(selisih_ke_2, 4) if selisih_ke_2 is not None else None,
    })

print("\n" + "="*78)
print("RINGKASAN UJI SIGNIFIKANSI: HDMC vs 10 PEMBANDING, PER DATASET")
print("="*78)
signifikansi_rows = []
if not df_sig.empty:
    for ds in DATASET_ORDER:
        sub_sig = df_sig[df_sig["dataset"] == ds]
        # PENTING: harus p<0.05 DAN effect!=negligible bareng, baru dibilang
        # "signifikan". Effect size "small/medium/large" saja tanpa p<0.05
        # BUKAN kemenangan signifikan (baru diperbaiki, sebelumnya salah hitung
        # 1 kasus: softfailure vs Baseline-RandomForest, p=0.08 lolos ambang effect
        # tapi p tidak signifikan).
        signifikan = sub_sig["p_value"] < 0.05
        menang = sub_sig[(sub_sig["cliffs_delta"] > 0) & (sub_sig["effect"] != "negligible") & signifikan]
        kalah  = sub_sig[(sub_sig["cliffs_delta"] < 0) & (sub_sig["effect"] != "negligible") & signifikan]
        setara = sub_sig[~sub_sig.index.isin(menang.index) & ~sub_sig.index.isin(kalah.index)]
        print(f"\n[{ds.upper()}] dari {len(sub_sig)} pembanding:")
        print(f"  HDMC menang signifikan : {len(menang)}  -> {list(menang['candidate_b'])}")
        print(f"  Setara (negligible)    : {len(setara)}  -> {list(setara['candidate_b'])}")
        if len(kalah) > 0:
            print(f"  HDMC kalah signifikan  : {len(kalah)}  -> {list(kalah['candidate_b'])}")
        signifikansi_rows.append({
            "dataset": ds, "total_pembanding": len(sub_sig),
            "menang_signifikan_jumlah": len(menang), "menang_signifikan_daftar": ", ".join(menang["candidate_b"]),
            "setara_jumlah": len(setara), "setara_daftar": ", ".join(setara["candidate_b"]),
            "kalah_signifikan_jumlah": len(kalah), "kalah_signifikan_daftar": ", ".join(kalah["candidate_b"]),
        })


## 3. Pola keunggulan HDMC vs Imbalance Ratio — klaim utama naskah

Klaim inti HDMC: keunggulan makin jelas seiring dataset makin timpang (IR
naik). Kalau jumlah "menang signifikan" naik monoton mengikuti urutan IR,
klaim ini terbukti dari data, bukan cuma narasi. Kalau tidak monoton, ini
perlu didiskusikan ulang sebelum ditulis sebagai klaim di naskah.

In [ ]:
ir_map = {"softfailure": 9.4, "hardfailure": 18.6, "otdr": 33.4, "sor": 558.5}
print("="*78)
print("TEMUAN KUNCI (pola vs Imbalance Ratio)")
print("="*78)
for ds in sorted(DATASET_ORDER, key=lambda d: ir_map.get(d, 0)):
    if df_sig.empty:
        break
    sub_sig = df_sig[df_sig["dataset"] == ds]
    n_menang = len(sub_sig[(sub_sig["cliffs_delta"] > 0) & (sub_sig["effect"] != "negligible")])
    print(f"  IR={ir_map.get(ds, 0):>6.1f}  {ds:<12}  HDMC menang signifikan atas {n_menang}/{len(sub_sig)} pembanding")

print("\nInterpretasi: kalau jumlah 'menang signifikan' naik seiring IR, ini")
print("mengonfirmasi klaim utama HDMC (keunggulan makin jelas di kondisi makin")
print("ekstrem). Kalau tidak monoton, itu perlu didiskusikan lagi sebelum ditulis")
print("sebagai klaim di naskah.")

# Gabungkan peringkat + signifikansi + IR jadi satu tabel "Ringkasan Temuan"
df_peringkat = pd.DataFrame(peringkat_rows)
df_signifikansi_ringkas = pd.DataFrame(signifikansi_rows)
df_ringkasan_temuan = df_peringkat.merge(df_signifikansi_ringkas, on="dataset", how="left")
df_ringkasan_temuan["imbalance_ratio"] = df_ringkasan_temuan["dataset"].map(ir_map)
df_ringkasan_temuan = df_ringkasan_temuan.sort_values("imbalance_ratio")
kolom_urut = ["dataset", "imbalance_ratio", "peringkat_HDMC", "total_model", "selisih_ke_posisi_2",
              "total_pembanding", "menang_signifikan_jumlah", "menang_signifikan_daftar",
              "setara_jumlah", "setara_daftar", "kalah_signifikan_jumlah", "kalah_signifikan_daftar"]
df_ringkasan_temuan = df_ringkasan_temuan[kolom_urut]


## 4. Kontribusi meta-learner — ringkasan hasil ablasi (6 kondisi, 2-treatment)

Enam kondisi: (1) backbone tunggal None+CE, (2) backbone tunggal
SMOTETomek+CE, (3) backbone tunggal None+Focal, (4) backbone tunggal
SMOTETomek+Focal, (5) multi-view voting (tanpa meta), (6) HDMC penuh
(proposed). Kalau kondisi 6 signifikan lebih baik dari kondisi lain,
itu bukti kontribusi. Kalau tidak signifikan di suatu dataset, itu
temuan jujur yang tetap ditulis apa adanya.

In [ ]:
ablasi_path = os.path.join(TAB_DIR, "ablasi_komponen_hdmc.csv")
sig_ablasi_path = os.path.join(TAB_DIR, "uji_signifikansi_ablasi_komponen.csv")

df_ablasi_sheet = pd.DataFrame()

if os.path.exists(ablasi_path):
    df_ablasi = pd.read_csv(ablasi_path)
    print("Skor 6 kondisi ablasi (macro-F1 mean):")
    print(df_ablasi.to_string(index=False))
    df_ablasi_sheet = df_ablasi.copy()
else:
    print(f"[lewati] {ablasi_path} tidak ditemukan")

if os.path.exists(sig_ablasi_path):
    df_sig_ablasi = pd.read_csv(sig_ablasi_path)
    print("\nSignifikansi tiap kondisi vs HDMC Proposed:")
    print(df_sig_ablasi.to_string(index=False))
    df_sig_ablasi["signifikan_beda"] = df_sig_ablasi["p_value"].apply(
        lambda p: "SIGNIFIKAN" if p < 0.05 else "tidak signifikan")
    if not df_ablasi_sheet.empty:
        df_ablasi_sheet = df_ablasi_sheet.merge(
            df_sig_ablasi[["dataset", "candidate_b", "p_value", "cliffs_delta", "effect", "signifikan_beda"]]
            .rename(columns={"candidate_b": "kondisi"}),
            on=["dataset", "kondisi"], how="left")
else:
    print(f"[lewati] {sig_ablasi_path} tidak ditemukan")

## 5. Dampak penggantian backbone CNN-BiLSTM -> CNN-BiGRU

Perbandingan HDMC hasil Part 2 (backbone lama, CNN-BiLSTM) vs Part 3
(backbone CNN-BiGRU, keputusan setelah justifikasi ulang menemukan
CNN-BiLSTM tidak konsisten menang di kriteria manapun). Angka Part 2 diambil
dari histori percakapan (bukan file, sekadar rujukan), angka Part 3 dari
file `<dataset>_hdmc_final_metrics.json` yang aktual.

In [ ]:
part2_lama = {
    "sor": 0.9130, "otdr": 0.9537, "hardfailure": 0.9483, "softfailure": 0.9893,
}

print("="*78)
print("PART 2 (CNN-BiLSTM) vs PART 3 (CNN-BiGRU)")
print("="*78)
print(f"{'Dataset':<14}{'Part 2':>10}{'Part 3':>10}{'Selisih':>10}")
backbone_rows = []
for ds in DATASET_ORDER:
    hdmc_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
    with open(hdmc_path) as f:
        part3_score = json.load(f)["macro_f1_mean"]
    part2_score = part2_lama.get(ds, float("nan"))
    print(f"{ds:<14}{part2_score:>10.4f}{part3_score:>10.4f}{part3_score-part2_score:>+10.4f}")
    backbone_rows.append({
        "dataset": ds, "part2_CNN_BiLSTM": round(part2_score, 4),
        "part3_CNN_BiGRU": round(part3_score, 4), "selisih": round(part3_score - part2_score, 4),
    })
df_backbone_sheet = pd.DataFrame(backbone_rows)

print("\nCatatan: Part 2 pakai protokol lama (5-fold/3-repeat, tanpa BAcc/MCC")
print("lengkap), Part 3 pakai protokol final (10-fold/1-repeat). Perbandingan")
print("ini indikatif soal arah perubahan, bukan perbandingan apple-to-apple")
print("protokol yang identik.")


## 6. Gabungkan semua ke satu file Excel (4 sheet baru + Petunjuk)

Menambahkan 4 sheet baru ke `HASIL_KONSOLIDASI_HDMC_vs_PEMBANDING.xlsx` yang
sudah dibuat di Langkah 1:

- **Petunjuk** — daftar isi tiap sheet (ditaruh paling depan)
- **Ringkasan Temuan** — peringkat HDMC + signifikansi + IR, satu tabel
- **Ablasi Meta-Learner** — kontribusi meta-learner per dataset (dari nb05)
- **Backbone Part2 vs Part3** — dampak ganti CNN-BiLSTM ke CNN-BiGRU

In [ ]:
from openpyxl import load_workbook

out_path = os.path.join(TAB_DIR, "HASIL_KONSOLIDASI_HDMC_vs_PEMBANDING.xlsx")

df_petunjuk = pd.DataFrame([
    {"Sheet": "Petunjuk", "Isi & Kegunaan": "Daftar isi ini."},
    {"Sheet": "Ringkasan Semua", "Isi & Kegunaan": "Semua model x semua dataset dalam satu tabel (44 baris)."},
    {"Sheet": "Sor / Otdr / Hardfailure / Softfailure", "Isi & Kegunaan":
        "Detail per dataset: metrik evaluasi, waktu training/inferensi, F1 per kelas asli dataset itu. Nilai terbaik di-bold."},
    {"Sheet": "Uji Signifikansi", "Isi & Kegunaan":
        "Wilcoxon + Cliff's delta, HDMC vs tiap pembanding, per dataset (p_value, cliffs_delta, effect)."},
    {"Sheet": "Ringkasan Temuan", "Isi & Kegunaan":
        "Peringkat HDMC, jumlah menang/setara/kalah signifikan, disortir dari IR terendah ke tertinggi."},
    {"Sheet": "Ablasi Meta-Learner", "Isi & Kegunaan":
        "Skor 3 kondisi (backbone tunggal, multi-view voting, HDMC penuh) + signifikansi kontribusi meta-learner."},
    {"Sheet": "Backbone Part2 vs Part3", "Isi & Kegunaan":
        "Dampak numerik penggantian backbone CNN-BiLSTM -> CNN-BiGRU, per dataset."},
])

wb = load_workbook(out_path)

def tulis_sheet(nama, df):
    if nama in wb.sheetnames:
        del wb[nama]
    ws = wb.create_sheet(nama)
    ws.append(list(df.columns))
    for _, row in df.iterrows():
        ws.append(list(row))

if not df_ringkasan_temuan.empty:
    tulis_sheet("Ringkasan Temuan", df_ringkasan_temuan)
if not df_ablasi_sheet.empty:
    tulis_sheet("Ablasi Meta-Learner", df_ablasi_sheet)
if not df_backbone_sheet.empty:
    tulis_sheet("Backbone Part2 vs Part3", df_backbone_sheet)
tulis_sheet("Petunjuk", df_petunjuk)

# Pindahkan "Petunjuk" ke paling depan
wb.move_sheet("Petunjuk", offset=-(wb.sheetnames.index("Petunjuk")))

wb.save(out_path)
print(f"Tersimpan lengkap (Petunjuk + Ringkasan Temuan + Ablasi + Backbone): {out_path}")
print(f"Total sheet sekarang: {wb.sheetnames}")